# Tutorial: Define molecular transition
- Preparation
- Define a transition for SMILESStringNode
- Generation with custom transition

## Background
This section provides background information to aid understanding, and is not directly related to the actual Transition implementation.

In [ ]:
# Define benzene as rdkit Mol object

from rdkit import Chem
from rdkit.Chem import Draw

benzene = Chem.MolFromSmiles("c1ccccc1")
Draw.MolToImage(benzene)

In [ ]:
# Define benzene as SMILESStringNode

from chemtsv3.node import SMILESStringNode
benezene_node = SMILESStringNode(string="c1ccccc1") # Same as SMILESStringNode(string="c1ccccc1", parent=None, last_prob=None, last_action=None)
Draw.MolToImage(benezene_node.mol()) # Subclasses of MolNode provide a mol() method that returns an RDKit Mol object.

In [ ]:
from rdkit.Chem import AllChem
from IPython.display import display

# Apply SMIRKS rule
# Implementation details are not essential for understanding, and can be safely skipped.
def apply_smirks(mol, smirks) -> list[str]:
    rxn = AllChem.ReactionFromSmarts(smirks)
    Chem.Kekulize(mol, clearAromaticFlags=True)
    results = [] # list of SMILES
    for ps in rxn.RunReactants((mol,)):
        for p in ps:
            try:
                p = Chem.RemoveHs(p)
                smiles = Chem.MolToSmiles(p, canonical=True)
                results.append(smiles)
            except:
                continue
    results = list(set(results)) # remove duplicates
    return results

# Test
# For other SMIRKS rule examples, see "data/smirks/example.txt"
append_n = "[*;!H0:1]>>[*:1]-N" # Append -N
append_f = "[*;!H0:1]>>[*:1]-F" # Append -F
insert_o = "[*:1]~[*:2]>>[*:1]O[*:2]" # Insert O

results = apply_smirks(benzene, append_n) + apply_smirks(benzene, append_f) + apply_smirks(benzene, insert_o)
for smiles in results:
    print(smiles)
    display(Draw.MolToImage(Chem.MolFromSmiles(smiles)))

## Define a transition for SMILESStringNode
A transition class returns a list of child nodes for a given node, optionally with the probability (`last_prob`) and / or the last action (`last_action`).

We recommend implementing a transition by inheriting from `TemplateTransition`, which already has `top_p`, `filters` and `logger` as parameters, and automatically normalizes `last_prob` of child nodes.

In [ ]:
from chemtsv3.transition import TemplateTransition

class ExampleTransition(TemplateTransition):
    def __init__(self, smirks_rules: list[str], top_p=None, filters=None, logger=None):
        self.smirks_rules = smirks_rules
        super().__init__(filters=filters, top_p=top_p, logger=logger) # Call __init__() of TemplateTransition to use those parameters
    
    # Define transition here
    def _next_nodes_impl(self, node: SMILESStringNode) -> list[SMILESStringNode]: # Input: initial node / Output: list of resulting nodes
        initial_mol = node.mol()
        smiles_list = []
        for smirks in self.smirks_rules:
            smiles_list += apply_smirks(initial_mol, smirks)
        smiles_list = list(set(smiles_list)) # Remove duplicates
        
        results = []
        for smiles in smiles_list:
            child_node = SMILESStringNode(string=smiles, parent=node, last_prob=1) # last_prob will be automatically normalized in TemplateTransition
            results.append(child_node)
            
        return results

In [ ]:
# Test ExampleTransition
smirks_rules = [
"[*;!H0:1]>>[*:1]-C", # append -C
"[*;!H0:1]>>[*:1]-Br", # append -Br
"[*;!H0;!H1:1]>>[*:1]=O", # append =O (not active for benzene)
"[C:1]>>[N:1]", # change C to N
"[*:1]@[*:2]>>([*:1].[*:2])" # delete cyclic bond
]
transition = ExampleTransition(smirks_rules=smirks_rules)
children = transition.next_nodes(benezene_node) # All transition classes have next_nodes(): overriding _next_nodes_impl() in TemplateTransition defines next_nodes(), which returns the final child nodes after filters, top-p selection, and probability normalization.

for c in children:
    smiles = c.key() # same as c.string
    print(smiles)
    display(Draw.MolToImage(Chem.MolFromSmiles(smiles)))

## Generation with custom transition
For YAML workflow, define the transition in a `***.py` file and place it in the `transition` directory. (You can find the same `ExampleTransition` as above in `transition/example_transition.py`.)

In YAML file, you can specify the transition class and its `__init__` arguments like this (see `config/tutorial/1c_example.yaml`.):
```
transition_class: ExampleTransition
transition_args:
  smirks_rules:
    - "[*;!H0:1]>>[*:1]-C"
    - "[*;!H0:1]>>[*:1]-Br"
    - "[*;!H0;!H1:1]>>[*:1]=O"
    - "[C:1]>>[N:1]"
    - "[*:1]@[*:2]>>([*:1].[*:2])"
  filters:
    - filter_class: ValidityFilter
    - filter_class: RadicalFilter
```

In [ ]:
import os
from chemtsv3.utils import conf_from_yaml, generator_from_conf

# Generate molecules with ExampleReward

repo_root = "../../"
yaml_path = "config/tutorial/1c_example.yaml" # Specify the yaml path. Open the file to see setting options.
yaml_path = os.path.join(repo_root, yaml_path)

conf = conf_from_yaml(yaml_path)
generator = generator_from_conf(conf, base_dir=repo_root)
generator.generate(time_limit=conf.get("time_limit"), max_generations=conf.get("max_generations"))
generator.plot(**conf.get("plot_args"))